# 05 — Project 2: Kindle Review Sentiment Analysis

Runnable companion to **note 18**. 12,000 Amazon Kindle book reviews — messier text, a harder
target, and a first model that scores **58%**.

**This notebook is really about diagnosis.** Getting a bad number is easy; knowing which of
six causes to fix first is the skill.

| Section | |
|---|---|
| 1. Load & explore | |
| 2. Ratings → binary (and the decision hidden in it) | |
| 3. Production-grade cleaning | |
| 4. The naive model → 58% | |
| 5. Diagnosis: six causes, measured one at a time | |
| 6. The fixed pipeline | |
| 7. Error analysis | |
| 8. The diagnostic checklist | |

## 0. Setup

In [ ]:
# !pip install pandas numpy scikit-learn nltk beautifulsoup4 lxml matplotlib

import re, time
import numpy as np
import pandas as pd
import nltk

for pkg in ["stopwords", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from bs4 import BeautifulSoup

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

RANDOM_STATE = 42
print("ready")

## 1. Load & explore

Amazon Kindle Store reviews, May 1996 – July 2014. A 12,000-row subset of the 5-core set.
Source: <https://jmcauley.ucsd.edu/data/amazon/> — credit and licence to the original authors.

In [ ]:
PATH = 'all_kindle_review.csv'

data = pd.read_csv(PATH)
print(data.shape)
print(list(data.columns))
data.head(3)

In [ ]:
df = data[['reviewText', 'rating']].copy()      # the only two columns we need

print("shape        :", df.shape)
print("missing      :", df.isnull().sum().to_dict())
print()
print(df['rating'].value_counts().sort_index())

In [ ]:
df['review_length'] = df['reviewText'].str.split().str.len()
print(df['review_length'].describe().round(1))
print()
print("Much longer than SMS (project 1 averaged ~15 words).")
print("That matters: TF-IDF and averaging behave differently on long text.")

In [ ]:
for r in [1, 5]:
    print(f"--- example {r}-star review ---")
    print(df[df['rating'] == r]['reviewText'].iloc[0][:320])
    print()

## 2. Ratings → binary, and the decision hidden in it

In [ ]:
df['rating_original'] = df['rating']
df['rating'] = df['rating'].apply(lambda x: 0 if x < 3 else 1)

print(df['rating'].value_counts())
print()
print(pd.crosstab(df['rating_original'], df['rating']))

### ⚠️ Where do the 3-star reviews go?

Here they are counted as **positive**. That is a *choice*, and it directly damages the model:

- A 3-star review is genuinely **neutral** — *"it was okay, a bit slow in the middle"*.
- Its language overlaps heavily with both 4-star and 2-star reviews.
- **2,000 of 12,000 rows (17%)** are these ambiguous cases, labelled positive.

**You are asking the model to learn a distinction that is not really in the text.** Remember
this when the accuracy comes out at 58% in §4.

In [ ]:
print("3-star reviews, as labelled positive:")
for t in df[df['rating_original'] == 3]['reviewText'].head(3):
    print("  -", t[:150])
print()
print("Read them. Would YOU call these positive?")

In [ ]:
baseline = df['rating'].value_counts(normalize=True).max()
print(f"majority-class baseline: {baseline:.4f}")
print("Any model scoring below this is worse than a constant guess.")

## 3. Production-grade cleaning

Real reviews contain HTML from scraping, URLs, email addresses and inconsistent whitespace.

> **Order matters.** Do HTML **first** — if you strip `<` and `>` with the special-character
> regex first, BeautifulSoup has nothing left to parse. The naive ordering below reproduces
> the common mistake; §3.3 fixes it.

### 3.1 🐛 First, the `''` vs `' '` bug

In [ ]:
s = "great book,loved it.highly recommend"

print("replace with ''  :", repr(re.sub('[^a-zA-Z]', '',  s)))
print("replace with ' ' :", repr(re.sub('[^a-zA-Z]', ' ', s)))
print()
print("Empty string GLUES words together, manufacturing junk tokens ('bookloved')")
print("that appear once each and pollute the vocabulary. Always substitute a SPACE.")

In [ ]:
# and note the space INSIDE the character class
s2 = "great book loved it"
print("[^a-z0-9]  (no space in class):", repr(re.sub('[^a-z0-9]',  '', s2)))
print("[^a-z0-9 ] (space in class)   :", repr(re.sub('[^a-z0-9 ]', '', s2)))
print()
print("Without the space in the class, your existing spaces are deleted too")
print("and the whole review becomes ONE enormous token.")

### 3.2 The URL regex, decoded

```
r'(http|https|ftp|ssh)://[\w_-]+(\.[\w_-]+)+\S*'
 └──── scheme ──────┘  └─ domain ─┘└─ .tld  ─┘└ rest
```

In [ ]:
url_re   = r'(http|https|ftp|ssh)://[\w_-]+(\.[\w_-]+)+\S*'
email_re = r'\S+@\S+\.\S+'

t = "Read more at https://amazon.co.uk/dp/B001?ref=x or email me at bob.smith@gmail.com now"
print("original:", t)
print("no URLs :", re.sub(url_re, '', t))
print("no mails:", re.sub(email_re, '', re.sub(url_re, '', t)))

### 3.3 Why BeautifulSoup rather than a regex for HTML

In [ ]:
html = '<p class="a>b">Great <b>book</b> &amp; a fast read<br/>'

print("regex  :", repr(re.sub('<.*?>', '', html)))
print("bs4    :", repr(BeautifulSoup(html, 'lxml').get_text()))
print()
print("The regex breaks on the '>' inside the attribute and leaves &amp; unresolved.")

### 3.4 The full cleaning function

In [ ]:
STOP = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_review(text, stop=STOP, keep_digits=True):
    text = str(text)
    text = BeautifulSoup(text, 'lxml').get_text()             # (1) HTML FIRST
    text = re.sub(url_re,   ' ', text)                        # (2) URLs
    text = re.sub(email_re, ' ', text)                        # (3) emails
    text = text.lower()                                       # (4) lowercase
    pattern = '[^a-z0-9 ]' if keep_digits else '[^a-z ]'
    text = re.sub(pattern, ' ', text)                         # (5) specials -> SPACE
    words = [w for w in text.split() if w not in stop]        # (6) stopwords
    words = [lemmatizer.lemmatize(w) for w in words]          # (7) lemmatize
    return ' '.join(words)                                    # (8) collapse whitespace

demo = 'This book was NOT good!! See <b>more</b> at https://x.com/y -- 3/5 stars.'
print("RAW  :", demo)
print("CLEAN:", clean_review(demo))

**Look at what just happened to `NOT`.** It is in the default stopword list, so
`"was NOT good"` became `"good"`. **The sentence now looks positive.** Hold that thought —
it is diagnosis cause #4.

In [ ]:
# speed up lemmatization with a cache -- most words repeat thousands of times
from functools import lru_cache
lemmatize_cached = lru_cache(maxsize=None)(lemmatizer.lemmatize)

t0 = time.time()
df['clean_naive'] = df['reviewText'].apply(clean_review)
print(f"cleaned {len(df)} reviews in {time.time()-t0:.1f}s")

df[['reviewText', 'clean_naive']].head(3)

In [ ]:
empty = (df['clean_naive'].str.len() == 0).sum()
print("empty after cleaning:", empty)
print("vocabulary size     :", len({w for t in df['clean_naive'] for w in t.split()}))

## 4. The naive model → 58%

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_naive'], df['rating'], test_size=0.20,
    random_state=RANDOM_STATE, stratify=df['rating'])

bow   = CountVectorizer()                       # NO max_features -- the full vocabulary
tfidf = TfidfVectorizer()

X_train_bow = bow.fit_transform(X_train)
X_test_bow  = bow.transform(X_test)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print("X_train_bow:", X_train_bow.shape)
print(f"{X_train_bow.shape[1]:,} features for {X_train_bow.shape[0]:,} samples "
      "-- MORE FEATURES THAN SAMPLES")

In [ ]:
dense_mb = X_train_bow.shape[0] * X_train_bow.shape[1] * 8 / 1e9
print(f"dense float64 would be ~{dense_mb:.2f} GB")
print(f"the sparse matrix is   ~{X_train_bow.data.nbytes/1e6:.1f} MB")
print()
print("DO NOT call .toarray() on this. sklearn's text models accept sparse input directly.")

In [ ]:
# GaussianNB needs dense input -- this is part of why it is the wrong choice here
gnb_bow   = GaussianNB().fit(X_train_bow.toarray(),   y_train)
gnb_tfidf = GaussianNB().fit(X_train_tfidf.toarray(), y_train)

acc_gnb_bow   = accuracy_score(y_test, gnb_bow.predict(X_test_bow.toarray()))
acc_gnb_tfidf = accuracy_score(y_test, gnb_tfidf.predict(X_test_tfidf.toarray()))

print(f"GaussianNB + BoW    : {acc_gnb_bow:.4f}")
print(f"GaussianNB + TF-IDF : {acc_gnb_tfidf:.4f}")
print(f"baseline (all-positive): {baseline:.4f}")
print()
print("WORSE THAN GUESSING. Do not shrug -- diagnose it.")

In [ ]:
print(pd.DataFrame(confusion_matrix(y_test, gnb_bow.predict(X_test_bow.toarray())),
                   index=['actual neg','actual pos'], columns=['pred neg','pred pos']))
print()
print(classification_report(y_test, gnb_bow.predict(X_test_bow.toarray()),
                            target_names=['negative','positive']))

## 5. Diagnosis — six causes, measured one at a time

We now change **one thing at a time** and record the accuracy after each. This is the
important part of the notebook.

In [ ]:
scores = {'naive (GaussianNB, everything wrong)': acc_gnb_bow}

def report(label, acc):
    scores[label] = acc
    print(f"{label:48} {acc:.4f}")

print(f"{'baseline (always predict positive)':48} {baseline:.4f}")
report('naive: GaussianNB + full-vocabulary BoW', acc_gnb_bow)

### 🔴 Cause 1 — `GaussianNB` is the wrong model for count features

`GaussianNB` assumes each feature is **normally distributed**. Word counts are not: they are
discrete and spiked at zero. Fitting a bell curve to that gives nonsense probabilities.

In [ ]:
col = np.asarray(X_train_bow[:, 5].todense()).ravel()
print("distribution of one word-count feature:")
print(pd.Series(col).value_counts().head())
print()
print("Mostly 0, occasionally 1. Nothing like a bell curve.")

In [ ]:
acc = accuracy_score(y_test, MultinomialNB().fit(X_train_bow, y_train).predict(X_test_bow))
report('fix 1: MultinomialNB instead of GaussianNB', acc)

**The single biggest win available.**

```
   GaussianNB     -> continuous, normally-distributed features (height, temperature)
   MultinomialNB  -> counts (BoW, TF-IDF)          <- use this for text
   BernoulliNB    -> binary presence/absence       <- good with binary=True
```

In [ ]:
for name, clf, X_tr, X_te in [
        ('MultinomialNB + BoW',    MultinomialNB(), X_train_bow,   X_test_bow),
        ('MultinomialNB + TF-IDF', MultinomialNB(), X_train_tfidf, X_test_tfidf),
        ('BernoulliNB   + BoW',    BernoulliNB(),   X_train_bow,   X_test_bow),
        ('LogisticRegr  + TF-IDF', LogisticRegression(max_iter=1000), X_train_tfidf, X_test_tfidf)]:
    a = accuracy_score(y_test, clf.fit(X_tr, y_train).predict(X_te))
    print(f"{name:28} {a:.4f}")

### 🟠 Cause 2 — no `max_features`: more features than samples

In [ ]:
for mf in [1000, 5000, 10000, None]:
    v  = TfidfVectorizer(max_features=mf)
    Xtr, Xte = v.fit_transform(X_train), v.transform(X_test)
    a = accuracy_score(y_test, LogisticRegression(max_iter=1000).fit(Xtr, y_train).predict(Xte))
    print(f"max_features={str(mf):6} -> {Xtr.shape[1]:6} features, accuracy {a:.4f}")

### 🟠 Cause 3 — the 3-star problem

In [ ]:
df_nonneutral = df[df['rating_original'] != 3].copy()
print("rows after dropping neutrals:", len(df_nonneutral))
print(df_nonneutral['rating'].value_counts())

Xa, Xb, ya, yb = train_test_split(
    df_nonneutral['clean_naive'], df_nonneutral['rating'],
    test_size=0.2, random_state=RANDOM_STATE, stratify=df_nonneutral['rating'])

v = TfidfVectorizer(max_features=5000)
a = accuracy_score(yb, LogisticRegression(max_iter=1000)
                      .fit(v.fit_transform(Xa), ya).predict(v.transform(Xb)))
report('fix 3: drop 3-star neutrals (+ fixes 1-2)', a)

### 🟡 Cause 4 — negation was removed with the default stopword list

In [ ]:
negations = {'not','no','nor','never','none','cannot',
             "don't","doesn't","didn't","isn't","wasn't","aren't","weren't",
             "won't","wouldn't","couldn't","shouldn't","can't","hasn't","haven't",
             'but','however','although'}

print("negation words inside NLTK's default English list:")
print(sorted(w for w in STOP if w in negations))

STOP_KEEP_NEG = STOP - negations
print(f"\ndefault: {len(STOP)}   curated: {len(STOP_KEEP_NEG)}")
print()
print("before:", clean_review("this book was not good at all"))
print("after :", clean_review("this book was not good at all", stop=STOP_KEEP_NEG))

In [ ]:
t0 = time.time()
df['clean_neg'] = df['reviewText'].apply(lambda t: clean_review(t, stop=STOP_KEEP_NEG))
print(f"re-cleaned in {time.time()-t0:.1f}s")

dfn = df[df['rating_original'] != 3]
Xa, Xb, ya, yb = train_test_split(dfn['clean_neg'], dfn['rating'],
                                  test_size=0.2, random_state=RANDOM_STATE,
                                  stratify=dfn['rating'])

v = TfidfVectorizer(max_features=5000)
a = accuracy_score(yb, LogisticRegression(max_iter=1000)
                      .fit(v.fit_transform(Xa), ya).predict(v.transform(Xb)))
report('fix 4: keep negations (+ fixes 1-3)', a)

### 🟡 Cause 5 — no n-grams, so `"not good"` is never a feature

In [ ]:
for rng in [(1,1), (1,2), (1,3)]:
    v  = TfidfVectorizer(max_features=5000, ngram_range=rng)
    Xtr, Xte = v.fit_transform(Xa), v.transform(Xb)
    acc = accuracy_score(yb, LogisticRegression(max_iter=1000).fit(Xtr, ya).predict(Xte))
    bigrams = [f for f in v.get_feature_names_out() if ' ' in f][:5]
    print(f"ngram_range={rng} -> {acc:.4f}   sample bigrams: {bigrams}")

In [ ]:
v = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=3)
a = accuracy_score(yb, LogisticRegression(max_iter=1000, class_weight='balanced')
                      .fit(v.fit_transform(Xa), ya).predict(v.transform(Xb)))
report('fix 5: + bigrams, min_df, class_weight', a)

### 🟢 Cause 6 — sentiment is simply harder than spam

Spam detection is keyword spotting (`free`, `win`, `claim`). Sentiment requires handling:

- **negation** — "not good"
- **sarcasm** — "oh, brilliant, another cliffhanger"
- **comparison** — "better than the last one, which was awful"
- **mixed opinions** — "great story, terrible editing"

**90%+ is realistic for spam. 80–85% is a good result for review sentiment with
bag-of-words methods.** Calibrate your expectations to the task.

## 6. The fixed pipeline

In [ ]:
df2 = df[df['rating_original'] != 3].copy()

X_tr, X_te, y_tr, y_te = train_test_split(
    df2['clean_neg'], df2['rating'], test_size=0.2,
    random_state=RANDOM_STATE, stratify=df2['rating'])

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=3)),
    ('clf',   LogisticRegression(max_iter=1000, class_weight='balanced',
                                 random_state=RANDOM_STATE)),
])

pipe.fit(X_tr, y_tr)
y_pred = pipe.predict(X_te)

final = accuracy_score(y_te, y_pred)
print(f"FINAL accuracy: {final:.4f}   (started at {acc_gnb_bow:.4f})")
print(f"improvement   : {final - acc_gnb_bow:+.4f}")
print()
print(classification_report(y_te, y_pred, target_names=['negative','positive']))

In [ ]:
print(pd.DataFrame(confusion_matrix(y_te, y_pred),
                   index=['actual neg','actual pos'], columns=['pred neg','pred pos']))

In [ ]:
summary = pd.DataFrame({'accuracy': pd.Series(scores)}).round(4)
summary['vs baseline'] = (summary['accuracy'] - baseline).round(4)
summary

**Six changes, a large jump — and not one of them was a fancier algorithm.**

### 6.1 What did the model learn? (always check)

In [ ]:
names = pipe['tfidf'].get_feature_names_out()
coefs = pipe['clf'].coef_[0]

print("most POSITIVE features:")
print("  ", [names[i] for i in np.argsort(coefs)[-20:][::-1]])
print()
print("most NEGATIVE features:")
print("  ", [names[i] for i in np.argsort(coefs)[:20]])

**Do these look like sentiment words?** If you see author names, book titles or series
names near the top, your model is memorising *products* rather than learning *sentiment* —
it will not generalise to new books.

## 7. Error analysis — worth more than any hyperparameter sweep

In [ ]:
proba = pipe.predict_proba(X_te)[:, 1]
X_te_reset, y_te_reset = X_te.reset_index(drop=True), y_te.reset_index(drop=True)

wrong = np.where(y_pred != y_te_reset.values)[0]
print(f"{len(wrong)} misclassified out of {len(y_te_reset)}")

# the most confidently wrong predictions -- these teach the most
confidence = np.abs(proba - 0.5)
worst = wrong[np.argsort(confidence[wrong])[::-1][:8]]

for i in worst:
    actual = 'positive' if y_te_reset.iloc[i] else 'negative'
    pred   = 'positive' if y_pred[i]          else 'negative'
    print(f"\nactual={actual}  predicted={pred}  p(pos)={proba[i]:.3f}")
    print("  ", X_te_reset.iloc[i][:220])

Read those. Common patterns you will find:

- **Sarcasm** — positive words, negative meaning.
- **Mixed reviews** — "loved the plot, hated the ending".
- **Plot description** — a review describing a *dark, tragic* story in negative vocabulary
  while actually praising the book.
- **Comparison** — "much better than the awful first book".

None of these is fixable by tuning. They need a model that understands sequence and
context — RNN, or better, a transformer.

### 7.1 Calibration — where is the model uncertain?

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 4))
plt.hist(proba[y_te_reset == 0], bins=40, alpha=0.6, label='actual negative')
plt.hist(proba[y_te_reset == 1], bins=40, alpha=0.6, label='actual positive')
plt.axvline(0.5, color='k', ls='--', lw=1, label='decision threshold')
plt.xlabel('predicted P(positive)')
plt.ylabel('count')
plt.title('Prediction confidence by true class')
plt.legend()
plt.tight_layout()
plt.show()

Overlap around 0.5 is where the errors live. A **large** overlap means the classes are
not cleanly separable with these features — which is exactly what the 3-star reviews caused
before we dropped them.

## 8. The diagnostic checklist

When a text model underperforms, work down this list **in order**:

```
 (1) MODEL/FEATURE MISMATCH   GaussianNB on counts? MultinomialNB on negatives?  <- biggest wins
 (2) LABELS                   Are the classes actually separable in the text?
                              Neutral rows forced into a binary label?
 (3) BASELINE                 Are you beating "always predict the majority class"?
 (4) FEATURE COUNT            More features than samples? Set max_features / min_df.
 (5) CLEANING                 Did you delete the signal? (negations, digits, emoji)
 (6) N-GRAMS                  Does the task need local word order?
 (7) LEAKAGE                  Fitted the vectorizer before splitting? (note 16)
 (8) IMBALANCE                Checked per-class precision/recall, not just accuracy?
 (9) ONLY NOW                 Try a different or bigger model.
```

**Almost every real fix lives in steps 1–6.** Reaching for a bigger model first is the
classic beginner move and usually the least effective one.

---

## Exercises

1. Re-run the six fixes in a **different order**. Does the ranking of their impact change?
2. Try **average Word2Vec** on this dataset (notebook 03). Reviews are long and nuanced —
   does it beat TF-IDF here, unlike in project 1?
3. Build the **three-class** version (neg / neutral / pos). Which pairs does the confusion
   matrix say the model confuses, and does that match your intuition?
4. Compare `LinearSVC`, `RandomForest` and `GradientBoosting` on the final features.
5. Add `sublinear_tf=True` to the vectorizer. Does it help on these long documents?
6. Take the 20 most confidently-wrong predictions and categorise them by failure type
   (sarcasm / mixed / plot-description / comparison). Which is most common?

---

## You have finished the course

Back to the [index](../notes/00-README-NLP-Index.md) — and try the twelve self-test questions
in Part 4 now that you have the answers.

```
   You are here --> RNN / LSTM / GRU --> Attention --> Transformers --> BERT / GPT
                    (word order,        (which words   (parallel,      (contextual
                     real sequences)     matter where)  scalable)       embeddings)
```